## Install libraries

In [1]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [2]:
embedding_model = OllamaEmbeddings(model="nomic-embed-text:latest")
llm = ChatOllama(model="deepseek-r1:1.5b")

## Step 1a - Indexing (Document Ingestion)

In [4]:
video_id = "W-7h6XHXecA" # only the ID, not full URL
from youtube_transcript_api import YouTubeTranscriptApi

ytt_api = YouTubeTranscriptApi()
ytt_api.fetch(video_id)

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='hi guys have ever wanted to create your', start=0.799, duration=3.801), FetchedTranscriptSnippet(text="own Chrome extension but didn't know", start=2.76, duration=3.999), FetchedTranscriptSnippet(text='where to start imagine being able to add', start=4.6, duration=3.999), FetchedTranscriptSnippet(text='new features to your browser and get', start=6.759, duration=4.0), FetchedTranscriptSnippet(text='things done quicker by creating Your Own', start=8.599, duration=4.2), FetchedTranscriptSnippet(text='Chrome extension you can take control of', start=10.759, duration=3.96), FetchedTranscriptSnippet(text='your browser and make it work the way', start=12.799, duration=4.32), FetchedTranscriptSnippet(text="you want in this video I'm going to show", start=14.719, duration=4.72), FetchedTranscriptSnippet(text='you how you can use CH GPT to create a', start=17.119, duration=4.361), FetchedTranscriptSnippet(text='fully functioning Chrome 

In [8]:
ytt_api = YouTubeTranscriptApi()
fetched_transcript = ytt_api.fetch(video_id)
transcript = []
# is iterable
for snippet in fetched_transcript:
   # print(snippet.text)
    transcript.append(snippet.text)

# indexable
last_snippet = fetched_transcript[-1]

# provides a length
snippet_count = len(fetched_transcript)
transcript

['hi guys have ever wanted to create your',
 "own Chrome extension but didn't know",
 'where to start imagine being able to add',
 'new features to your browser and get',
 'things done quicker by creating Your Own',
 'Chrome extension you can take control of',
 'your browser and make it work the way',
 "you want in this video I'm going to show",
 'you how you can use CH GPT to create a',
 'fully functioning Chrome extension and',
 "whether you're a beginner or have some",
 'coding experience this tutorial will',
 'guide you through the process step by',
 "step so I'm brand from Learners and",
 "let's get started",
 '[Music]',
 "first let's talk about what a Chrome",
 'extension is and why you might want to',
 'create one a Chrome extension is a small',
 'software program that customizes your',
 'browsing experience in the Google Chrome',
 'web browser it can add new features',
 'modify web pages or perform action based',
 "on certain events okay we're going to",
 'create a Chrome exten

## Step 1b - Indexing (Text Splitting)

In [12]:
updated_transcript = " ".join(transcript)
updated_transcript

"hi guys have ever wanted to create your own Chrome extension but didn't know where to start imagine being able to add new features to your browser and get things done quicker by creating Your Own Chrome extension you can take control of your browser and make it work the way you want in this video I'm going to show you how you can use CH GPT to create a fully functioning Chrome extension and whether you're a beginner or have some coding experience this tutorial will guide you through the process step by step so I'm brand from Learners and let's get started [Music] first let's talk about what a Chrome extension is and why you might want to create one a Chrome extension is a small software program that customizes your browsing experience in the Google Chrome web browser it can add new features modify web pages or perform action based on certain events okay we're going to create a Chrome extension in just two steps the first step is to create the files needed for the extension and we're g

In [13]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([updated_transcript])

In [14]:
len(chunks)

8

In [16]:
chunks[1]

Document(metadata={}, page_content="modify web pages or perform action based on certain events okay we're going to create a Chrome extension in just two steps the first step is to create the files needed for the extension and we're going to use CH GPT to create the files for our extension and with these files we can launch the extension on the Chrome browser so to create the files needed for our extension let's go to chat GPD I'm switching to GPD 4 which has a feature to create files in just one click okay now we need to enter prompt to create our files I've already prepared a prompt to get all the files in just one click you can just copy the prompt from the description below and and then paste it here so here enter what your extension is about and how it should work I want to create a YouTube bookmark extension and with this extension whenever I watch a YouTube video I want to be able to bookmark it or save any specific point and later if I open it I want the video to play from that 

## Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [17]:
#embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embedding_model)

In [18]:
vector_store.index_to_docstore_id

{0: '9dbc131a-c495-41b6-a3e4-fd2f8958abf0',
 1: '15cd15cc-a6f6-4f6f-952d-0f7ab6162997',
 2: 'c5c51965-6c3d-4a1b-b668-10daa09d6a5f',
 3: 'c9c79213-8e3b-4cb1-bc78-c9dade1bd27d',
 4: 'c16505c5-a853-4cb0-87f4-faed08b30b5f',
 5: 'd3d7ff38-effa-4989-90df-fc9a32f7a206',
 6: 'a8424cee-61bc-4925-82e9-424ef984314c',
 7: '72b725ce-0615-4813-9e6e-90eacef43350'}

In [ ]:
vector_store.get_by_ids(['2436bdb8-3f5f-49c6-8915-0c654c888700'])

[Document(id='2436bdb8-3f5f-49c6-8915-0c654c888700', metadata={}, page_content='demas establish to support this podcast please check out our sponsors in the description and now let me leave you with some words from edskar dykstra computer science is no more about computers than astronomy is about telescopes thank you for listening and hope to see you next time')]

## Step 2 - Retrieval

In [19]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [20]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001AC40723E00>, search_kwargs={'k': 4})

In [21]:
retriever.invoke('What is deepmind')

[Document(id='72b725ce-0615-4813-9e6e-90eacef43350', metadata={}, page_content="of the buttons you want so once you've entered that press enter and chat jpt will generate the file for the new look of our extension so let's download it move it to the extensions folder and click replace all right now that we have replaced the old file let's see how it works so let's click on the extension and there you go we have a new attractive extension so that's it guys this is how you can create a Chrome extension using CH GPD and not just that CH GPD can help you create more incredible things if you want to learn how to create an entire website just by using AI you can check out this video and if you like this video make sure you give it a thumbs up and don't forget to subscribe to website learners for more cool videos like this one thanks for watching I'll see you in the next video Until then take care bye-bye [Music]"),
 Document(id='9dbc131a-c495-41b6-a3e4-fd2f8958abf0', metadata={}, page_conten

## Step 3 - Augmentation

In [22]:
#llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)

In [23]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [24]:
question          = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs    = retriever.invoke(question)

In [25]:
retrieved_docs

[Document(id='a8424cee-61bc-4925-82e9-424ef984314c', metadata={}, page_content="looks like okay once we got the extension let's check if it's working I'll go to YouTube video and play it hi guys so in today's video I'm going to show you how you can make your own a now I want to bookmark this video at this specific time so let's open the extension and click on save current time time now anytime later if I go to the extension and reopen The Bookmark that I said before so just drag and drop the audio here let's listen to it you can see the video plays from that specific timestamp now we have successfully added the Chrome extension and this is how looks so let's say you want to change the appearance and make it look more attractive we can do that too with the help of chbd just describe how you want the extension to look like you can mention the color scheme and the style of the buttons you want so once you've entered that press enter and chat jpt will generate the file for the new look of 

In [26]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"looks like okay once we got the extension let's check if it's working I'll go to YouTube video and play it hi guys so in today's video I'm going to show you how you can make your own a now I want to bookmark this video at this specific time so let's open the extension and click on save current time time now anytime later if I go to the extension and reopen The Bookmark that I said before so just drag and drop the audio here let's listen to it you can see the video plays from that specific timestamp now we have successfully added the Chrome extension and this is how looks so let's say you want to change the appearance and make it look more attractive we can do that too with the help of chbd just describe how you want the extension to look like you can mention the color scheme and the style of the buttons you want so once you've entered that press enter and chat jpt will generate the file for the new look of our extension so let's download it move it to the extensions folder and click\n

In [27]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [28]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      looks like okay once we got the extension let's check if it's working I'll go to YouTube video and play it hi guys so in today's video I'm going to show you how you can make your own a now I want to bookmark this video at this specific time so let's open the extension and click on save current time time now anytime later if I go to the extension and reopen The Bookmark that I said before so just drag and drop the audio here let's listen to it you can see the video plays from that specific timestamp now we have successfully added the Chrome extension and this is how looks so let's say you want to change the appearance and make it look more attractive we can do that too with the help of chbd just describe how you want the extension to look like you can mention the color scheme and the style of the bu

## Step 4 - Generation

In [29]:
answer = llm.invoke(final_prompt)
print(answer.content)

<think>
Okay, so I'm trying to figure out whether the topic of nuclear fusion is discussed in the provided video. Let me go through my thoughts step by step.

First, I remember that the video title mentions "nuclear fusion" explicitly. The title says, "Nuclear Fusion discussed in this video," which makes me think yes, it's definitely covered. But wait, maybe I'm overcomplicating it. Let me read the transcript again carefully to confirm.

Looking at the transcript, the first paragraph talks about checking if the extension works and then explaining how to bookmark a video later. Then there's a description of creating an extension using CH GPT. It mentions something about adding new features based on events in Chrome. The main focus seems to be on extending Chrome's functionality rather than discussing nuclear fusion.

So, while the topic is mentioned at the beginning, the actual content around the middle discusses how extensions work and how you can modify Chrome's behavior by customizin

## Building a Chain

In [30]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [31]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [32]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [33]:
parallel_chain.invoke('who is Demis')

{'context': "you already have an icon you can replace this file with your own icon but make sure you name it as icon.png if you don't have an icon you can simply search on the internet to find one so I'm going to search for YouTube icons similarly you can search for the icon you want right click on the image and click save image as go to the extensions folder and here we have the old icon to replace the new icon here enter icon.png click save and click yes to replace okay so now we've added the icon and all the errors should be fixed to check it let's go back and if we click retry you can see our extension is active and if we go to this extension icon you can see our extension has been successfully added to Chrome now we can pin the extension to access it easily and if we click on it this is how our extension looks like okay once we got the extension let's check if it's working I'll go to YouTube video and play it hi guys so in today's video I'm going to show you how you can make your 

In [34]:
parser = StrOutputParser()

In [35]:
main_chain = parallel_chain | prompt | llm | parser

In [36]:
main_chain.invoke('Can you summarize the video')

"<think>\nOkay, so the user has provided a transcript of a video about creating a Chrome extension using CH GPT. They're asking me to summarize the video. Let me try to understand what's happening here.\n\nFirst, I see that the video starts with someone explaining how they want their Chrome extension to work: making it bookmarkable and save specific points so they can resume playback later. The person is a tech enthusiast looking to automate parts of their browser experience.\n\nThey go into detail about features like saving timestamps and renaming extensions. Then there's an example of what the extension should look like, with buttons and color schemes. It includes steps on how to create the code for the extension using CH GPT and then distributing it via Google Chrome.\n\nThe user also adds a note that they can explore more by creating their own websites or apps if they have coding skills. They wrap up by encouraging viewers to leave questions in case something isn't clear.\n\nSo, pu